In [3]:
import baostock as bs
import pandas as pd

lg = bs.login()
print(f"登录状态: {lg.error_code}, {lg.error_msg}")

rs = bs.query_hs300_stocks()
hs300_df = rs.get_data()
print(f"成分股数量: {len(hs300_df)}")
print(hs300_df.head())

bs.logout()

login success!
登录状态: 0, success
成分股数量: 300
   updateDate       code code_name
0  2026-10-12  sh.600000      浦发银行
1  2026-10-12  sh.600009      上海机场
2  2026-10-12  sh.600010      包钢股份
3  2026-10-12  sh.600011      华能国际
4  2026-10-12  sh.600015      华夏银行
logout success!


In [4]:
import time

stock_codes = hs300_df['code'].tolist()
print(f"待获取股票数量: {len(stock_codes)}")

all_data = []
start_date = '2020-01-01'
end_date = '2024-12-31'

lg = bs.login()  # 只登录一次

for i, code in enumerate(stock_codes):
    rs = bs.query_history_k_data_plus(
        code,
        "date,code,open,high,low,close,volume,amount,pctChg",
        start_date=start_date,
        end_date=end_date,
        frequency="d",
        adjustflag="2"
    )
    df = rs.get_data()
    all_data.append(df)
    
    if (i + 1) % 20 == 0:
        print(f"[{i+1}/{len(stock_codes)}] 已完成")
    time.sleep(0.2)  # 缩短到0.2秒

bs.logout()  # 最后统一登出

print("全量获取完成")

待获取股票数量: 300
login success!
[20/300] 已完成
[40/300] 已完成
[60/300] 已完成
[80/300] 已完成
[100/300] 已完成
[120/300] 已完成
[140/300] 已完成
[160/300] 已完成
[180/300] 已完成
[200/300] 已完成
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[WinError 10053] 你的主机中的软件中止了一个已建立的连接。
接收数据异常，请稍后再试。
[220

In [25]:
final_df = pd.concat(all_data, ignore_index=True)
final_df.to_csv('../data/processed_stock_data_raw.csv', index=False)
print(f"已保存: {len(final_df)} 条")

已保存: 344636 条


In [7]:
import time

all_codes = set(hs300_df['code'].tolist())
got_codes = set(final_df['code'].unique())
missing_codes = list(all_codes - got_codes)
print(f"需要补跑: {len(missing_codes)} 只")

# 重新登录
lg = bs.login()
print(f"登录状态: {lg.error_code}")

for i, code in enumerate(missing_codes):
    # 每30只重新登录一次，防止长连接超时
    if i > 0 and i % 30 == 0:
        bs.logout()
        time.sleep(2)
        bs.login()
        print(f"重新登录 (已补 {i} 只)")
    
    rs = bs.query_history_k_data_plus(
        code,
        "date,code,open,high,low,close,volume,amount,pctChg",
        start_date='2020-01-01',
        end_date='2024-12-31',
        frequency="d",
        adjustflag="2"
    )
    df = rs.get_data()
    if len(df) > 0:
        all_data.append(df)
    
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{len(missing_codes)}] 补跑中")
    time.sleep(1)  # 关键：改成1秒

bs.logout()
print("补跑完成")

需要补跑: 97 只
login success!
登录状态: 0
[10/97] 补跑中
[20/97] 补跑中
[30/97] 补跑中
logout success!
login success!
重新登录 (已补 30 只)
[40/97] 补跑中
[50/97] 补跑中
[60/97] 补跑中
logout success!
login success!
重新登录 (已补 60 只)
[70/97] 补跑中
[80/97] 补跑中
[90/97] 补跑中
logout success!
login success!
重新登录 (已补 90 只)
logout success!
补跑完成


In [4]:
import pandas as pd

test_df = pd.concat(all_data, ignore_index=True)
print(f"总记录数: {len(test_df)}")
print(f"列名: {test_df.columns.tolist()}")
print(test_df.head(10))

总记录数: 6060
列名: ['date', 'code', 'open', 'high', 'low', 'close', 'volume', 'amount', 'pctChg']
         date       code          open          high           low  \
0  2020-01-02  sh.600000  9.1233886900  9.2477652800  9.1087561500   
1  2020-01-03  sh.600000  9.1965513900  9.2404490100  9.1233886900   
2  2020-01-06  sh.600000  9.1599700400  9.2550815500  9.0868073400   
3  2020-01-07  sh.600000  9.1526537700  9.2185002000  9.1160724200   
4  2020-01-08  sh.600000  9.0794910700  9.1087561500  8.9624307500   
5  2020-01-09  sh.600000  9.0648585300  9.0941236100  9.0355934500   
6  2020-01-10  sh.600000  9.0502259900  9.0868073400  9.0063283700   
7  2020-01-13  sh.600000  9.0721748000  9.0794910700  9.0063283700   
8  2020-01-14  sh.600000  9.0721748000  9.2843466300  9.0648585300   
9  2020-01-15  sh.600000  9.0794910700  9.1087561500  8.9624307500   

          close    volume          amount     pctChg  
0  9.1233886900  51629079  647446166.0000   0.808400  
1  9.2185002000  38018810

In [8]:
import pandas as pd

final_df = pd.concat(all_data, ignore_index=True)
print(f"总记录数: {len(final_df)}")
print(f"股票数量: {final_df['code'].nunique()}")

final_df.to_csv('../data/processed_stock_data_raw.csv', index=False)
print("完整数据已保存")

总记录数: 344636
股票数量: 298
完整数据已保存


In [11]:
counts = final_df.groupby('code').size()
print(f"最少记录数: {counts.min()}")
print(f"最多记录数: {counts.max()}")
print(f"平均记录数: {counts.mean():.1f}")
print(f"记录数少于1000的股票:")
print(counts[counts < 1000])

最少记录数: 2
最多记录数: 1212
平均记录数: 1156.5
记录数少于1000的股票:
code
sh.600905    865
sh.600938    656
sh.600941    725
sh.601059    468
sh.601136    491
sh.601728    815
sh.601825    816
sh.601868    790
sh.603296    340
sh.605117    899
sh.605499    875
sh.688041    579
sh.688047    614
sh.688072    657
sh.688082    758
sh.688183    936
sh.688223    710
sh.688271    573
sh.688303    836
sh.688472    380
sh.688506    481
sz.001391      2
sz.301165    514
sz.301236    681
sz.301269    589
sz.301308    584
dtype: int64


In [ ]:
#剔除记录数过少的股票
counts = final_df.groupby('code').size()
keep_codes = counts[counts >= 500].index.tolist()
print(f"保留 {len(keep_codes)} 只股票")

final_df = final_df[final_df['code'].isin(keep_codes)].reset_index(drop=True)
print(f"剩余记录数: {len(final_df)}")
print(f"剩余股票数: {final_df['code'].nunique()}")

保留 292 只股票
剩余记录数: 342474
剩余股票数: 292


In [26]:
# 数据清洗函数
def clean_data(df):
    numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount', 'pctChg']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.sort_values(['code', 'date']).reset_index(drop=True)
    df[numeric_cols] = df.groupby('code')[numeric_cols].ffill()
    return df

In [27]:
# 技术指标函数
import pandas as pd
import numpy as np

def add_indicators(group):
    group = group.copy()
    # MACD
    ema_fast = group['close'].ewm(span=12).mean()
    ema_slow = group['close'].ewm(span=26).mean()
    group['macd'] = ema_fast - ema_slow
    group['macd_signal'] = group['macd'].ewm(span=9).mean()
    # RSI
    delta = group['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    group['rsi'] = 100 - (100 / (1 + gain / loss))
    # 布林带
    ma20 = group['close'].rolling(20).mean()
    std20 = group['close'].rolling(20).std()
    group['boll_upper'] = ma20 + 2 * std20
    group['boll_lower'] = ma20 - 2 * std20
    # 波动率
    group['log_ret'] = np.log(group['close'] / group['close'].shift(1))
    group['volatility_60'] = group['log_ret'].rolling(60).std() * np.sqrt(252)
    return group

In [28]:
final_df = clean_data(final_df)
print(f"清洗后: {len(final_df)} 条")

清洗后: 344636 条


In [29]:
final_df = final_df.groupby('code', group_keys=False).apply(add_indicators)
print(f"指标计算完成，列数: {len(final_df.columns)}")

指标计算完成，列数: 15


In [30]:
final_df.to_csv('../data/processed_stock_data.csv', index=False)
print("已保存")

已保存
